In [ ]:
# Ячейка 1: Постановка
"""
## Бюджет логистической готовности

Primal: распределение бюджета на топливо (X) и запчасти (Y).
Эффективность: 50 ед/млн на топливо, 80 ед/млн на запчасти.

Ограничения (млн руб):
- Общий бюджет: X + Y <= 10
- Топливо (мин): X >= 3
- Запчасти (мин): Y >= 2
- Пропорция: X <= 2Y

Задание:
1. Решить прямую задачу
2. Записать dual с учетом разных знаков ограничений
3. Какое ограничение самое ценное
4. Что будет при увеличении бюджета на 1 млн
"""

In [ ]:
# Ячейка 2: Решение
import numpy as np
from scipy.optimize import linprog

# Primal: max Z = 50X + 80Y -> min -Z
c = [-50, -80]

# Приводим все к <=
A = [[1, 1],    # X + Y <= 10
     [-1, 0],    # -X <= -3
     [0, -1],    # -Y <= -2
     [1, -2]]    # X - 2Y <= 0
b = [10, -3, -2, 0]

result = linprog(c, A_ub=A, b_ub=b, bounds=(0, None), method='highs')

X, Y = result.x
Z = -result.fun

print(f"Топливо X = {X:.2f} млн")
print(f"Запчасти Y = {Y:.2f} млн")
print(f"Эффективность Z = {Z:.2f}")

# Slack
slack = b - A @ result.x
names = ['Бюджет', 'Топливо мин', 'Запчасти мин', 'Пропорция']
for n, s in zip(names, slack):
    print(f"{n}: запас = {s:.2f}")

In [ ]:
# Ячейка 3: Dual (прямо из linprog)
"""
Двойственные переменные доступны через result.ineqlin.marginals
(в зависимости от версии scipy)
"""

# Ручной dual
c_dual = [10, -3, -2, 0]  # b из primal
A_dual = [[-1, 1, 0, -1],   # -y1 + y2 + 0 - y4 <= -50
          [-1, 0, 1, 2]]     # -y1 + 0 + y3 + 2y4 <= -80

b_dual = [-50, -80]
result_dual = linprog(c_dual, A_ub=A_dual, b_ub=b_dual, bounds=(0, None), method='highs')

print("Теневые цены:")
print(f"Бюджет: {result_dual.x[0]:.3f}")
print(f"Мин топливо: {result_dual.x[1]:.3f}")
print(f"Мин запчасти: {result_dual.x[2]:.3f}")
print(f"Пропорция: {result_dual.x[3]:.3f}")

In [ ]:
# Ячейка 4: Вывод
"""
Самое ценное ограничение - бюджет (наибольшая теневая цена).
+1 млн бюджета даст прирост эффективности на y1.
Минимальные ограничения имеют нулевую цену (не активны).
"""